## Addition Interp (2) !

This notebook is an attempt to explore how transformers learn to represent Fibonacci recurrence. In short, our question is : 

**"How do small transformers trained on addition data learn to add?"**

authored by : vorrjjard

##### Setup (Don't read, just run!)

In [140]:
import torch as t
import transformer_lens

from transformer_lens import HookedTransformer, ActivationCache, HookedTransformerConfig
from transformer_lens.train import HookedTransformerTrainConfig, train

import typing
from typing import Literal, List

from fib_interp_2.utils import generate_sample
from dataclasses import dataclass

import numpy as np

In [137]:
DEVICE = 'cuda' if t.cuda.is_available() else 'cpu'

##### 1. Data Generation

Related to Nanda's work on modular addition, we first start by creating a long tensor containing all our dataset examples.


In [ ]:
@dataclass
class dataConfig:
    min_output: int = 0 #@param
    n_digits_input = 3
    n_digits_output = 4
    max_output: int = 999 #@param
    n_samples: int = 1000
data_cfg = dataConfig()
print(data_cfg)

In [147]:
vocab = {str(i): i for i in range(10)}
vocab['='] = 10

Let's define a small vocabulary for our project. Hence, our `d_vocab = 10` 

In [ ]:
# 0 - 9, = : 10.

x = [vocab[char] for char in generate_sample(data_cfg)]

In [ ]:
# Generate samples, tokenize, and stack on top of a new batch dimension.

samples_tensor = t.stack(
    [t.tensor([vocab[char] for char in generate_sample(data_cfg)]) for n in range(0, data_cfg.n_samples)]
    ,dim=0)

##### 2. Model Configuration / Training

In [163]:
@dataclass
class Config(HookedTransformerConfig):
    d_model: int = 128
    debug: bool = True
    layer_norm_eps: float = 1e-5
    d_vocab: int = 10 # 0 through 9, comma equals
    init_range: float = 0.02
    n_ctx: int = 11
    d_head: int = 32
    d_mlp: int = 512
    n_heads: int = 4
    n_layers: int = 1
    device: str = DEVICE

cfg = Config()
print(cfg)

Config(d_model=128, d_head=32, n_layers=1, n_ctx=11, n_heads=4, d_mlp=512, d_vocab=10, device='cpu', use_attn_result=False, use_split_qkv_input=False, default_prepend_bos=True, positional_embedding_type='standard', n_key_value_heads=None, attn_only=False, gated_mlp=False, uses_rms_norm=False, eps=1e-05, layer_norm_folding=False, act_fn='relu', normalization_type='LN', num_experts=None, experts_per_token=None, final_rms=False, dtype=torch.float32, model_name='custom', use_attn_scale=True, attn_scale=np.float64(5.656854249492381), use_hook_mlp_in=False, use_attn_in=False, use_qk_norm=False, use_local_attn=False, ungroup_grouped_query_attention=False, original_architecture=None, from_checkpoint=False, checkpoint_index=None, checkpoint_label_type=None, checkpoint_value=None, tokenizer_name=None, window_size=None, attn_types=None, init_mode='gpt2', n_devices=1, attention_dir='causal', seed=None, initializer_range=np.float64(0.07071067811865475), init_weights=True, scale_attn_by_inverse_laye

In [139]:
@dataclass 
class trainConfig(HookedTransformerTrainConfig):
    batch_size : int = 128
    device : str = DEVICE
    num_epochs : int = 1000
    lr : float = 0.001
    seed : int = 42
    save_every : int = 100
    wand : bool = False

In [135]:
model = HookedTransformer(cfg)

TypeError: HookedTransformerTrainConfig.__init__() missing 2 required positional arguments: 'num_epochs' and 'batch_size'